This file acts as a replacement for main script in order to easily tweak different parts of code

In [ ]:
from data import load_data
import matplotlib.pyplot as plt
import numpy as np
from models.layer.initializer import Initializer
from rich.table import Table
from rich.console import Console
from models.layer.activation import Relu, Softmax
from models.layer import Layer
from scripts.train import create_and_train_model_from_config
from config.config import Config
from typing import Any
from utils import plot_all
import shutil
import yaml

In [ ]:
def write_config(field: str, value: Any):
    PATH = "../config/config.yaml"
    with open(PATH, "rt") as f:
        data = yaml.safe_load(f)
    f.close()

    data[field] = value
    with open(PATH, "wt") as f:
        data = yaml.safe_dump(data, f)
    f.close()


shutil.copy("../config/config.yaml.base", "../config/config.yaml")

In [ ]:
data = load_data()

Base Model

In [ ]:
model, logs = create_and_train_model_from_config(Config.read_yaml(), return_logs=True)
print(model.evaluate(X=data.X_train, y=data.y_train))
plot_all(model, logs)

Learning rate

In [ ]:
for lr in (1, 1e-2, 1e-5):
    cfg = Config.read_yaml()
    for layer in cfg.layers:
        layer.learning_rate = lr
    model, logs = create_and_train_model_from_config(cfg, return_logs=True)
    plot_all(model=model, logs=logs)
    plt.title(f"{lr = }")
    plt.savefig(f"../../report/images/q2_lr_{lr}.png")
    print(f"lr {lr} done")

Standarizing

In [ ]:
cfg = Config.read_yaml()
cfg.normalize = True
model, logs = create_and_train_model_from_config(cfg, return_logs=True)
print(model.evaluate(X=data.X_val, y=data.y_val))
plot_all(model=model, logs=logs)
plt.title(f"normalizing input")
plt.savefig(f"../../report/images/q2_normalize.png")
print(f"done")

In [ ]:
write_config(field="normalize", value=True)

Batch size

In [ ]:
for batch_size in (8, 128):
    cfg = Config.read_yaml()
    cfg.batch_size = batch_size
    plt.figure()
    model, logs = create_and_train_model_from_config(cfg, return_logs=True)
    plot_all(model=model, logs=logs)
    plt.title(f"{batch_size = }")
    plt.savefig(f"../../report/images/q2_bs_{batch_size}.png")
    print(f"bs {batch_size} done")
    print(model.evaluate(X=data.X_val, y=data.y_val))

In [ ]:
write_config("batch_size", 8)

Momentum

In [ ]:
cfg = Config.read_yaml()
for layer in cfg.layers:
    layer.momentum = 0.9
model, logs = create_and_train_model_from_config(cfg, return_logs=True)
plot_all(model=model, logs=logs)
plt.title(f"momento = 0.9")
plt.savefig(f"../../report/images/q2_momento.png")
print(model.evaluate(X=data.X_val, y=data.y_val))
print(model.evaluate(X=data.X_train, y=data.y_train))
print(f"done")

In [ ]:
cfg = Config.read_yaml()
for layer in cfg.layers:
    layer.momentum = 0.7
model, logs = create_and_train_model_from_config(cfg, return_logs=True)
plot_all(model=model, logs=logs)
plt.title(f"momento = 0.7")
plt.savefig(f"../../report/images/q2_momento_7.png")
print(model.evaluate(X=data.X_val, y=data.y_val))
print(model.evaluate(X=data.X_train, y=data.y_train))
print(f"done")

In [ ]:
write_config("momentum", 0.9)

Neuron count

In [ ]:
for size in (8, 16, 64):
    cfg.layers = Layer.get_layers(
        hidden_layer_sizes=[size],
        momentum=0.9,
        learning_rate=1e-3,
        regularization_factor=0,
    )

    model, logs = create_and_train_model_from_config(cfg, return_logs=True)
    plot_all(model=model, logs=logs)
    plt.title(f"{size = }")
    plt.savefig(f"../../report/images/q2_size_{size}.png")
    print("Train", model.evaluate(X=data.X_train, y=data.y_train))
    print("Val", model.evaluate(X=data.X_val, y=data.y_val))
    print(f"done {size}")

In [ ]:
write_config("layers_sizes", [64])

Layer Count

In [ ]:
cfg = Config.read_yaml()
cfg.layers = Layer.get_layers(
    hidden_layer_sizes=[16, 32],
    momentum=0.9,
    learning_rate=1e-3,
    regularization_factor=0,
)

model, logs = create_and_train_model_from_config(cfg, return_logs=True)
plot_all(model=model, logs=logs)
plt.title("two hidden layer")
plt.savefig(f"../../report/images/q2_two_layer.png")
print("Train", model.evaluate(X=data.X_train, y=data.y_train))
print("Val", model.evaluate(X=data.X_val, y=data.y_val))
print(f"done")

Overfit

In [ ]:
table = Table()
table.add_column("technique")
table.add_column("parameter")
table.add_column("train accuracy")
table.add_column("val accuracy")

plt.figure(figsize=(15, 10))
cfg = Config.read_yaml()

model, logs = create_and_train_model_from_config(cfg, return_logs=True)
table.add_row(
    "No overfit fix",
    "-",
    str(model.evaluate(X=data.X_train, y=data.y_train)),
    str(model.evaluate(X=data.X_val, y=data.y_val)),
)
print("Base Done")
for i, factor in enumerate((1e-2, 1e-3, 1e-4)):
    cfg.layers = Layer.get_layers(
        hidden_layer_sizes=[64],
        momentum=0.9,
        learning_rate=1e-3,
        regularization_factor=factor,
    )

    model, logs = create_and_train_model_from_config(cfg, return_logs=True)

    plt.subplot(2, 3, i + 1)

    plt.plot([i["train error"] for i in logs], label="train", lw=3, alpha=0.7)
    plt.plot([i["val error"] for i in logs], label="val", lw=3, alpha=0.7)
    plt.xlabel("Epoch")
    plt.ylabel("Error")
    plt.legend()
    plt.title(f"Regularization Factor = {factor}")
    table.add_row(
        "regularization",
        str(factor),
        str(model.evaluate(X=data.X_train, y=data.y_train)),
        str(model.evaluate(X=data.X_val, y=data.y_val)),
    )
    print("Regularization", i, "done")


for i, threshold in enumerate((3, 5, 10)):
    cfg = Config.read_yaml()
    cfg.early_stopping_threshold = threshold

    model, logs = create_and_train_model_from_config(cfg, return_logs=True)
    plt.subplot(2, 3, i + 1 + 3)

    plt.plot([i["train error"] for i in logs], label="train", lw=3, alpha=0.7)
    plt.plot([i["val error"] for i in logs], label="val", lw=3, alpha=0.7)
    plt.xlabel("Epoch")
    plt.ylabel("Error")
    plt.legend()
    plt.title(f"Early stopping {threshold}")
    table.add_row(
        "early stopping",
        str(threshold),
        str(model.evaluate(X=data.X_train, y=data.y_train)),
        str(model.evaluate(X=data.X_val, y=data.y_val)),
    )
    print("Early stopping", i, "done")
plt.savefig(f"../../report/images/q2_fix_overfit.png")
plt.show()
console = Console()
console.log(table)

In [ ]:
write_config("regularization_factor", 0.001)